# The Math of Meaning: Counting and Modeling Textual Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/UW-Madison-Digital-Scholarship-Hub/text-and-data/blob/main/04-math-of-meaning/The-Math-of-Meaning.ipynb)

How do you compare a thousand documents at once? Nobody can read them all side by side, but a computer can count them.

In this notebook, we'll turn text into numbers we can count and compare: **bag-of-words** representations, **TF-IDF**, and measures of **similarity**. We'll see how the size of a collection changes what we can find, what these methods leave out, and how they connect to the **embeddings** behind modern language AI.

You don't need any math background. Every idea starts with something you can picture, or hold in your hand, before it gets a name. Throughout, we'll ask you to **predict first, then run the code**. A wrong guess is often the most useful result you'll get today.

> **Important Step**

Run the cells in order from top to bottom. Each cell depends on the ones above it. You don't need a GPU for this notebook; Colab's default CPU runtime is fine. If your session restarts, choose **Runtime > Run before** on the cell you were working on to catch up.

In [ ]:
""" If you're working in Colab, run this cell to bring everything, including the
saved copies of our books, over from the GitHub repository. It's safe to re-run."""
import os # lets us check which folder we're in
if not os.path.exists('The-Math-of-Meaning.ipynb'): # already in the right folder? then there's nothing to do
    if not os.path.exists('text-and-data'): # only download the repository once
        !git clone https://github.com/UW-Madison-Digital-Scholarship-Hub/text-and-data.git
    %cd text-and-data/04-math-of-meaning

## Packages
We'll use a few Python packages to count and compare our texts:
- [**pandas**](https://pandas.pydata.org/): An open source data manipulation and analysis package
- [**NumPy**](https://numpy.org/): A package for calculating with lots of numbers at once
- [**scikit-learn**](https://scikit-learn.org/): A machine learning package, which we'll use to measure similarity
- [**nltk**](https://www.nltk.org/): The Natural Language Toolkit, which we'll use to get a collection of 500 documents
- [**matplotlib**](https://matplotlib.org/): A package for drawing charts

In [ ]:
# Packages to install if necessary (they all come preinstalled in Colab)
%pip install -q pandas numpy scikit-learn nltk matplotlib

import gzip # lets us read our compressed copies of the books
import json # lets us read saved embeddings
import random # lets us draw random samples of documents
import re # regular expression library, for finding words in text
from collections import Counter # counts things for us

import matplotlib.pyplot as plt
import nltk
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

BOOK_COLORS = ["#2a78d6", "#eb6834", "#1baf7a"] # blue, orange, aqua: the same colors as our last workshop

## Counting Before Coding

You can learn a lot from counting without writing any code. [Google Books Ngram Viewer](https://books.google.com/ngrams) charts how often words appear in millions of books.

**Predict first:** How have *save*, *salvation*, *print*, *fax*, and *message* changed since 1800? Sketch the curves for *save* and *fax* before opening the chart.

- [See the five words](https://books.google.com/ngrams/graph?content=save,salvation,print,fax,message&year_start=1800&year_end=2022&corpus=en&smoothing=3)

Is *save* one word? Saving a soul, saving money, and saving a file are the same five letters, so a single curve blends them. Counting short phrases can pull them apart:

- [See the phrases](https://books.google.com/ngrams/graph?content=save+the+file,save+your+work,save+money,save+souls,save+his+soul&year_start=1800&year_end=2022&corpus=en&smoothing=3)

**Discuss:** What is the curve actually counting? What are you inferring on top of it? Notice that changing *what we count* changed *what we could see*. We'll come back to that all session.

## A Bag of Tiles

Time to put the laptop aside. With your group and a Bananagrams set, try three rounds:

**Round 1: Into the bag.** Spell *stone*, then drop the tiles into a bag and shake it. Could someone looking in the bag tell you what the word was? Could it have been a different word?

**Round 2: Reveal one tile.** Your partner picks a secret word and reveals one tile at a time. How much does an **E** help you guess? How about a **Q**? Why the difference?

**Round 3: Tally and compare.** Write *stone*, *notes*, and *rock* as rows of letter counts, one column for each letter. Which two are the most alike? Are they the most alike in *meaning*?

Keep your tiles out. We're going to treat them as a tiny model of a collection of documents:

| On the table | In text analysis |
|---|---|
| Tile | Word |
| Spelled word | Document |
| All the words on the table | Collection |

One caution before we go on: letters don't mean anything on their own, and words do. The tiles will show us how the math *works*. We'll need real texts to see what it can tell us.

## Bags in Code

A **bag of words** is what you made in Round 3, one level up. We throw out the order and keep the counts, so each document becomes a row of numbers with one column for each word. The whole table is sometimes called a **document-term matrix**.

We'll write one function to build that table, and then use it at every size today, from a few tiles to 500 documents.

In [ ]:
# We'll create a function that finds the words in a text, as we did in our last workshop
def find_words(text):
    """Return a list of the lowercase words in a text, keeping contractions like "don't"."""
    return re.findall(r"[a-z]+(?:'[a-z]+)?", text.lower().replace("’", "'"))

# We'll create a function that turns documents into a table of counts
def bag_of_words(documents, tokenize=find_words):
    """Return a table with one row per document, one column per word, and counts in the cells."""
    # documents is a dictionary like {"name of document": "text of document"}
    counts = {name: Counter(tokenize(text)) for name, text in documents.items()}
    return pd.DataFrame.from_dict(counts, orient="index").fillna(0).astype(int).sort_index(axis=1)

First, the tiles. Here the "words" in our table are letters and the "documents" are words, so we tell the function to split each word into letters.

In [ ]:
my_tiles = ["stone", "notes", "rock"] # ✏️ add words from your table
letter_table = bag_of_words({word: word for word in my_tiles}, tokenize=list) # list splits a word into its letters
letter_table

Does this match the table you wrote by hand? Now let's do the same thing with sentences, where the columns are real words.

In [ ]:
sentences = {
    "A": "The dog bites the man.",
    "B": "The man bites the dog.",
    "C": "The dog sleeps.",
}
bag_of_words(sentences)

Sentences A and B have opposite meanings, but they have identical rows. To a bag of words, they're the same document, just as *stone* and *notes* are the same bag of tiles.

✏️ **Try it:** Can you write two sentences with very different meanings and identical rows? Compete with your neighbor for the biggest difference in meaning.

In [ ]:
my_sentences = {
    "mine": "She only said she loved him.", # ✏️ change this sentence
    "yours": "She said she only loved him.", # ✏️ change this sentence
}
bag_of_words(my_sentences)

## TF-IDF: What Makes a Document Distinctive?

Remember Round 2? An E barely helped, and a Q helped a lot. That intuition, that rare things tell you more, is the core of **TF-IDF**. It asks two questions about each word in each document:

1. **How much of this document is this word?** This is **term frequency** (TF).
2. **How few documents have this word?** This is **inverse document frequency** (IDF).

Let's start on the table. **Predict first:** Look at these words. Which letters best set *quiz* apart? Which letter on the table is the least useful for telling the words apart?

In [ ]:
table_words = ["stone", "notes", "rock", "quiz", "tile", "green", "banana", "river"] # ✏️ add or swap in your own words
tiles = bag_of_words({word: word for word in table_words}, tokenize=list)
tiles

**Question 1: How much of the document is this letter?** We divide each count by the number of letters in that word. Each "z" is a quarter of *quiz*, but it would be a much smaller share of a longer word.

In [ ]:
term_frequency = tiles.div(tiles.sum(axis=1), axis=0) # divide each count by the total for its row
term_frequency.round(2)

**Question 2: How few documents have this letter?** First we count how many words each letter appears in. Then we divide the size of the collection by that count, and take the logarithm.

In [ ]:
document_frequency = (tiles > 0).sum(axis=0) # in how many documents does each letter appear?
number_of_documents = len(tiles)

rarity = pd.DataFrame({
    "words containing it": document_frequency,
    "collection size / that": number_of_documents / document_frequency,
    "IDF (the log of that)": np.log(number_of_documents / document_frequency),
})
rarity.sort_values("IDF (the log of that)").round(2)

<details>
<summary><b>Why the logarithm?</b> (Optional: click to read)</summary>

A logarithm squashes big differences down to a manageable size. Without one, a word in 1 out of 1,000 documents would count 1,000 times as much as a word found in all of them, and it would swamp everything else. With one, its score is about 7.

It also gives us a neat result: if a word is in *every* document, then collection size / documents containing it = 1, and the log of 1 is exactly **0**. So a word found everywhere scores zero.
</details>

**Multiply the two together** and we have TF-IDF. Here's the whole thing as a single function that we can use on any table of counts.

In [ ]:
# We'll create a function that calculates TF-IDF for a table of counts
def tf_idf(counts):
    """Return a table of TF-IDF scores: how much of the document is this word, times how rare the word is."""
    counts = counts.loc[:, (counts > 0).any()] # ignore any words that aren't in these documents
    term_frequency = counts.div(counts.sum(axis=1), axis=0)
    document_frequency = (counts > 0).sum(axis=0)
    inverse_document_frequency = np.log(len(counts) / document_frequency)
    return term_frequency * inverse_document_frequency

# We'll create a function that shows the highest-scoring words in a document
def top_words(table, document, n=10):
    """Return the n words with the highest values in one document's row."""
    return table.loc[document].nlargest(n).round(4)

tile_scores = tf_idf(tiles)
top_words(tile_scores, "quiz", 4) # ✏️ try another word from your table

Did the top letters match your prediction? Notice that *i* scores lower than *q*, *u*, and *z*, because other words on the table have an *i* too. On tiles, these scores tell us about spelling and nothing else. So let's move on to real books.

*A note for later: if you use TF-IDF in `scikit-learn`, you'll get different numbers. Its default formula adds a little to every score, so words like "the" don't reach zero. There are several versions of TF-IDF, and that's another choice someone has made for you.*

### Eight Books

We'll use the eight books from our last workshop, which are already saved in this repository.

We've left the stop words in on purpose. Last time we chose which ones to remove by hand. Let's see what happens if we don't.

In [ ]:
BOOKS = {
    98: "A Tale of Two Cities",
    2701: "Moby-Dick",
    1342: "Pride and Prejudice",
    1260: "Jane Eyre",
    345: "Dracula",
    84: "Frankenstein",
    11: "Alice in Wonderland",
    1661: "Sherlock Holmes",
}

# We'll create a function that opens our saved copy of a book
def load_book(book_id):
    """Return the text of a saved book, without the Project Gutenberg header and license."""
    with gzip.open(f"../03-Tokenization/texts/{book_id}.txt.gz", "rt", encoding="utf-8") as f:
        text = f.read()
    start = re.search(r"\*\*\* START OF.*?\*\*\*", text).end()
    end = re.search(r"\*\*\* END OF", text).start()
    return text[start:end]

book_counts = bag_of_words({title: load_book(book_id) for book_id, title in BOOKS.items()})
print(f"{book_counts.shape[0]} books, {book_counts.shape[1]:,} different words, {book_counts.values.sum():,} words in total")

Here are the most common words in each book. Could you tell which book is which from these lists?

In [ ]:
pd.DataFrame({title: book_counts.loc[title].nlargest(8).index for title in BOOKS.values()}, index=range(1, 9))

Tools like [Voyant](https://voyant-tools.org/) make this kind of counting easy without any code. Paste in a text, or a link to one like [*Moby-Dick*](https://www.gutenberg.org/ebooks/2701.txt.utf-8), and you get a word cloud, frequency charts, and much more. It's a great place to start exploring, and you can take it with you. It also has to deal with the problem we can see here, which is why it comes with a stop word list.

**Predict first:** Pick a book you know. Write down the three words you think are the most distinctive to it. Then run the next cell.

In [ ]:
book_scores = tf_idf(book_counts)
pd.DataFrame({title: book_scores.loc[title].nlargest(8).index for title in BOOKS.values()}, index=range(1, 9))

No stop word list needed: "the" is in all eight books, so its score is exactly zero. What's left is mostly a cast of characters and a setting.

**Discuss:** Is that what you'd call what a book is *about*? *Moby-Dick* gets *whale*, but what happened to obsession, or fate? Why might names win?

Let's look at how the scores depend on the collection. **Predict first:** *Whale* is near the top for *Moby-Dick*. What happens to its score if we compare *Moby-Dick* with only one other book, *Dracula*?

In [ ]:
pair = book_counts.loc[["Moby-Dick", "Dracula"]] # ✏️ try a different second book
pair_scores = tf_idf(pair)

print("The word 'whale' appears this many times:")
print(pair["whale"].to_string())
print()
print("Top words for Moby-Dick against just this one book:")
print(list(top_words(pair_scores, "Moby-Dick", 8).index))
print()
print("Score for 'whale' here:", round(pair_scores.loc["Moby-Dick", "whale"], 5))
print("Score for 'whale' among all eight books:", round(book_scores.loc["Moby-Dick", "whale"], 5))

*Moby-Dick* didn't change, but the score did. A single *whale* in all of *Dracula* is enough to make it a word that's in "every" document of this tiny collection, and so its score falls to zero.

**TF-IDF doesn't measure anything about a book by itself. It measures a book against a collection.** Choosing the collection is part of the analysis. We'll come back to this soon.

## Similarity: Which Way Is It Pointing?

A row of numbers can also be drawn as an arrow. Imagine a map where we walk east once for each time a text says *whale*, and north once for each time it says *sea*. Each text's two counts land it at one spot on the map, and we can draw an arrow from the start to that spot.

**Predict first:** A short text and a long text have the same mix of words, and a third text of medium length has a very different mix. Which two are the most similar?

In [ ]:
# Made-up counts for three texts: (times it says "whale", times it says "sea")
texts = {"Long text": (8, 4), "Short text": (2, 1), "Different mix": (1, 6)}

fig, ax = plt.subplots(figsize=(6, 5))
for (name, (x, y)), color in zip(texts.items(), BOOK_COLORS):
    ax.annotate("", xy=(x, y), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color=color, linewidth=2 if name != "Short text" else 4, shrinkA=0, shrinkB=0))
    ax.text(x + 0.15, y + 0.15, f"{name} ({x}, {y})", color="#333333")
ax.set_xlim(0, 11)
ax.set_ylim(0, 7)
ax.set_title("Length is how much text; direction is the mix of words", loc="left")
ax.set_xlabel('Times the text says "whale"')
ax.set_ylabel('Times the text says "sea"')
ax.spines[["top", "right"]].set_visible(False) # remove the box around the chart
ax.grid(color="#e5e5e5")
ax.set_axisbelow(True)
plt.show()

pd.DataFrame(cosine_similarity(list(texts.values())), index=texts, columns=texts).round(2)

The short and long texts point in exactly the same direction, even though one is four times the length of the other. That's why we compare **direction** instead of position: a long book and a short book shouldn't look different just because of their length.

**Cosine similarity** is a score for how closely two arrows line up. For counts, it runs from **1** (the same direction) to **0** (nothing in common). The two arrows here have only 2 dimensions, but the math doesn't care: it works the same way with tens of thousands of words, even though nobody can picture that map.

Let's check it against your tiles from Round 3:

In [ ]:
# We'll create a function that compares every document in a table with every other document
def similarity_table(table):
    """Return a table of the cosine similarity between each pair of rows."""
    return pd.DataFrame(cosine_similarity(table), index=table.index, columns=table.index)

similarity_table(bag_of_words({word: word for word in ["stone", "notes", "rock"]}, tokenize=list)).round(2) # ✏️ try your own words

*Stone* and *notes* are a perfect match, and *stone* and *rock* share only an O. Keep that in mind, because it'll come back at the end.

Now for the books. **Predict first:** Which two of our eight books do you think are the most alike? Which two are the least alike? Discuss with a neighbor, and then compare your answer to the counts.

In [ ]:
similarity_table(book_counts).round(2).style.background_gradient(cmap="Blues", axis=None, vmin=0.8, vmax=1).format("{:.2f}")

Everything looks nearly the same! That's because most of each arrow's length comes from the same few words, like *the* and *of*. Let's try again with TF-IDF scores.

In [ ]:
book_similarity = similarity_table(book_scores)
book_similarity.round(3).style.background_gradient(cmap="Blues", axis=None, vmin=0, vmax=0.08).format("{:.3f}")

Now everything looks different from everything else! The same books, compared by the same measure, went from "nearly identical" to "almost nothing in common." We changed what we were counting, and that changed the answer. Neither one is simply correct.

What's the closest pair now? Does it match what you predicted? Let's ask what's behind that score.

In [ ]:
# We'll create a function that shows which words contribute most to the similarity of two documents
def shared_words(table, first, second, n=8):
    """Return the words that do the most to make two documents look alike."""
    # each word's score in one document times its score in the other
    return (table.loc[first] * table.loc[second]).nlargest(n)

print(list(shared_words(book_scores, "Pride and Prejudice", "Frankenstein").index))
print(list(shared_words(book_scores, "Pride and Prejudice", "Jane Eyre").index)) # ✏️ try another pair

Is *Pride and Prejudice* like *Frankenstein* because they're similar novels, or because both have a character named Elizabeth? The number can't tell us. To find out, we have to go back and look at which words are responsible, and then read.

## How Big Is the Collection?

Eight books isn't many. To see how the size of a collection matters, we'll use the **Brown Corpus**, a collection of 500 samples of American English from 1961, about 2,000 words each. Every sample was sorted into one of 15 genres when the collection was built, like *news*, *religion*, and *romance*. Those labels are handy, since they give us something to check our results against.

In [ ]:
nltk.download("brown", quiet=True) # download the Brown Corpus
from nltk.corpus import brown

brown_texts = {doc: " ".join(brown.words(doc)) for doc in brown.fileids()}
brown_genre = {doc: brown.categories(doc)[0] for doc in brown.fileids()}
brown_counts = bag_of_words(brown_texts)

print(f"{brown_counts.shape[0]} documents, {brown_counts.shape[1]:,} different words")
pd.Series(brown_genre).value_counts()

Pick a document from this collection. We'll start with `ca01`, a news story from Atlanta, and we'll score it inside collections of 10, 50, 100, and 500 documents. Each collection includes all of the documents from the smaller ones.

**Predict first (write it down!):**
- As the collection grows, will this document's top words change a little, or a lot?
- What will the document's nearest neighbor be? Will it become more or less sensible?
- Do you expect there to be a size where things stop changing?

In [ ]:
my_document = "ca01" # ✏️ try "cj01" (science), "cd01" (religion), "cg01" (essay), or "ck01" (fiction)
print(f"Genre: {brown_genre[my_document]}")
print(" ".join(brown.words(my_document)[:40]), "...")

# Build the same collections each time by shuffling once (with a fixed seed), then taking the first n
shuffled = random.Random(4).sample(list(brown_texts), 500)
shuffled.remove(my_document)

rows = []
for size in [10, 50, 100, 500]:
    collection = [my_document] + shuffled[:size - 1]
    scores = tf_idf(brown_counts.loc[collection])
    similarity = similarity_table(scores)[my_document].drop(my_document)
    neighbor = similarity.idxmax()
    rows.append({
        "collection size": size,
        "top words": ", ".join(top_words(scores, my_document, 6).index),
        "nearest neighbor": neighbor,
        "its genre": brown_genre[neighbor],
        "similarity": round(similarity.max(), 2),
    })

pd.set_option("display.max_colwidth", None) # show all of the top words
pd.DataFrame(rows).set_index("collection size")

**Discuss with a neighbor:** What did you predict correctly? What surprised you? Is this document's *neighbor* a good one? (You can read it with `" ".join(brown.words("ca06")[:100])`, replacing `ca06` with its ID.)

If you've got your tiles out: merge your words with another group's table. Which letters stop being rare? It's the same thing happening here.

That's one document, though. For a broader check, we can test the whole collection, using the genre labels. For every document, does its nearest neighbor have the same genre? We'll test this at several sizes, with several random samples at each size.

**Predict first:** What fraction of documents will have a neighbor of the same genre with all 500? And what fraction would we expect by pure chance?

In [ ]:
# We'll create a function that tests how often documents' nearest neighbors share their genre
def genre_match_rate(documents):
    """Return the fraction of documents whose nearest neighbor has the same genre, and the rate expected by chance."""
    similarity = cosine_similarity(tf_idf(brown_counts.loc[documents]))
    np.fill_diagonal(similarity, -1) # a document doesn't count as its own neighbor
    neighbors = similarity.argmax(axis=1)
    genres = [brown_genre[doc] for doc in documents]
    matches = np.mean([genres[i] == genres[j] for i, j in enumerate(neighbors)])
    group_sizes = pd.Series(genres).value_counts()
    chance = (group_sizes / len(documents) * (group_sizes - 1) / (len(documents) - 1)).sum()
    return matches, chance

rows = []
for size in [10, 50, 100, 250, 500]:
    # average over several random samples, except at 500, where there's only one possible sample
    samples = [random.Random(seed).sample(list(brown_texts), size) for seed in range(5 if size < 500 else 1)]
    results = [genre_match_rate(sample) for sample in samples]
    rows.append({"collection size": size, "same genre": np.mean([r[0] for r in results]), "by chance": np.mean([r[1] for r in results])})
match_rates = pd.DataFrame(rows).set_index("collection size")

ax = match_rates["same genre"].plot(figsize=(7, 4), color=BOOK_COLORS[0], linewidth=2, marker="o", markersize=8)
match_rates["by chance"].plot(ax=ax, color="#777777", linewidth=2, linestyle="--")
ax.text(515, match_rates["same genre"].iloc[-1], "Nearest neighbor", va="center", color="#333333")
ax.text(515, match_rates["by chance"].iloc[-1], "By chance", va="center", color="#333333")
ax.set_xlim(0, 620)
ax.set_ylim(0, 0.5)
ax.set_title("How often does a document's nearest neighbor share its genre?", loc="left")
ax.set_xlabel("Number of documents in the collection")
ax.set_ylabel("Share of documents")
ax.yaxis.set_major_formatter(lambda value, position: f"{value:.0%}")
ax.spines[["top", "right"]].set_visible(False) # remove the box around the chart
ax.grid(axis="y", color="#e5e5e5")
ax.set_axisbelow(True)
plt.show()

match_rates.round(2)

**Discuss:** Is that good? It's far better than chance, with no one telling the computer what a genre is. But most documents still have a neighbor from a different genre, and a bigger collection doesn't keep helping. More data isn't automatically more understanding.

Not all genres do equally well. Why might some be easier to find than others?

In [ ]:
all_documents = list(brown_texts)
similarity = cosine_similarity(tf_idf(brown_counts.loc[all_documents]))
np.fill_diagonal(similarity, -1) # a document doesn't count as its own neighbor
neighbors = similarity.argmax(axis=1)
genres = pd.Series([brown_genre[doc] for doc in all_documents])
neighbor_genres = pd.Series([brown_genre[all_documents[i]] for i in neighbors])

pd.DataFrame({
    "documents": genres.value_counts(),
    "share with same-genre neighbor": (genres == neighbor_genres).groupby(genres).mean().round(2),
}).sort_values("share with same-genre neighbor", ascending=False)

Think back to our eight books: TF-IDF loved character names. Every work of fiction has its own cast, so what might two stories have in common for this method to find? News stories, on the other hand, often share subjects and vocabulary.

Look at *fiction*, though. Many of its nearest neighbors are in *adventure* and *romance*, so is that a failure of the method, or a question about the labels? Someone decided in the 1960s where each of these texts belonged. Our "answer key" is also a set of human choices.

## What Counting Can't See: Embeddings

Remember *stone* and *rock*? To a bag of words, two different words are simply two different columns, so "car" and "automobile" have nothing in common. Word order is gone too.

Modern language AI uses a different kind of row of numbers, called an **embedding**. Instead of counting, a model learns to assign the numbers by studying how words are used across huge amounts of text. Words that keep similar company end up with similar numbers.

Let's see what one looks like. We've saved some embeddings from a small model in this folder, so nothing needs to be downloaded yet.

In [ ]:
with open("sample_data/saved_embeddings.json") as f:
    saved = json.load(f)
embeddings = {text: np.array(vector) for text, vector in saved["vectors"].items()}

sentence = "She picked up a heavy stone."
other = "She picked up a large rock."

print("As a bag of words (one column for each word):")
display(bag_of_words({"This sentence": sentence, "Another sentence": other}))

print(f"As an embedding: {len(embeddings[sentence])} numbers. Here are the first 12:")
print(embeddings[sentence][:12].round(3))

**Discuss:** In the bag of words, we can say exactly what every number means: "this sentence has one *stone*." What can we say about the embedding's numbers? (None of them have names, and hardly any are zero.) What did we gain, and what did we give up?

Numbers this long are hard to read, so let's turn each one into a color. Each stripe below is one word's embedding.

In [ ]:
words_to_show = ["stone", "rock", "notes", "banana"]
strips = np.array([embeddings[word] for word in words_to_show])

fig, ax = plt.subplots(figsize=(10, 2.5))
image = ax.imshow(strips, aspect="auto", cmap="RdBu_r", vmin=-0.2, vmax=0.2, interpolation="nearest")
ax.set_yticks(range(len(words_to_show)), words_to_show)
ax.set_xlabel("The 384 numbers in each embedding")
ax.set_title("Each word is a row of 384 numbers, shown as colors", loc="left")
fig.colorbar(image, ax=ax, label="Value")
plt.show()

Can you tell by eye which two are the most alike? Neither can we, and that's what the math is for.

**Predict first:** Back to your tiles. By letter counts, *stone* and *notes* were a perfect match, and *stone* and *rock* barely overlapped. What do you think the embedding will say? Rank the three pairs below, from the most to the least similar.

In [ ]:
# We'll create a function that scores two embeddings with the same cosine similarity we used for the books
def embedding_score(first, second):
    """Return the cosine similarity of two saved embeddings."""
    return cosine_similarity([embeddings[first]], [embeddings[second]])[0][0]

# ...and one that does the same thing for tile counts
def tile_score(first, second):
    """Return the cosine similarity of two words, based on their letter counts."""
    return cosine_similarity(bag_of_words({first: first, second: second}, tokenize=list))[0][1]

pd.DataFrame(
    [{"pair": f"{a} / {b}", "by letter counts": tile_score(a, b), "by embedding": embedding_score(a, b)}
     for a, b in [("stone", "notes"), ("stone", "rock"), ("stone", "banana")]]
).set_index("pair").round(2)

The embedding sees what the tiles couldn't: *stone* and *rock* are related. The math of the comparison hasn't changed at all. It's the same cosine similarity that we used on the books. Only the numbers going into it are different.

Notice, though, that *stone* and *banana* don't score zero. With embeddings, unrelated things can still score fairly high, so a score only means something next to other scores.

Do embeddings fix everything? **Predict first:** Which pairs below will the model think are the most alike?

In [ ]:
# We'll create a function that scores two sentences as a bag of words
def word_score(first, second):
    """Return the cosine similarity of two texts, based on their word counts."""
    return cosine_similarity(bag_of_words({"first": first, "second": second}))[0][1]

pairs = [
    ("I took my car to the mechanic.", "The automobile needed repairs at the garage."),
    ("I took my car to the mechanic.", "The chef cooked a lovely dinner."),
    ("The movie was good.", "The movie was not good."),
    ("The dog bit the man.", "The man bit the dog."),
    ("I sat on the river bank.", "I deposited money at the bank."),
]
pd.DataFrame(
    [{"first": a, "second": b, "by word counts": word_score(a, b), "by embedding": embedding_score(a, b)} for a, b in pairs]
).round(2)

The two car sentences share no meaningful words, and the only overlap the word counts can find is little words like *the* and *to*. The embedding gives them a much higher score than the sentence about the chef. But it still rates "The dog bit the man" and "The man bit the dog" as almost identical, and it says "good" and "not good" are close. Embeddings are a powerful way of capturing how words are used, which isn't the same thing as understanding what a sentence says.

### Try It Yourself
Now you can use the model itself. The next cell downloads a small model, [`all-MiniLM-L6-v2`](https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2) (about 90 MB), so give it a minute. It's free, and it runs right here in your notebook. If the download is slow, team up with a neighbor.

In [ ]:
%pip install -q sentence-transformers
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2") # downloads the model the first time

# We'll create a function that compares two texts both ways
def compare(first, second):
    """Print the word-count similarity and the embedding similarity of two texts."""
    vectors = model.encode([first, second])
    print(f"By word counts: {word_score(first, second):.2f}")
    print(f"By embedding:   {cosine_similarity(vectors)[0][1]:.2f}")

compare("She picked up a heavy stone.", "She picked up a large rock.")

Remember *save* from the start of the session? One Ngram curve blended all of its senses together. **Predict first:** Which of these will the first sentence be closest to?

In [ ]:
save_sentences = [
    "Jesus came to save sinners.", # ✏️ try your own word with several meanings
    "The preacher promised salvation.",
    "She wants to save money for college.",
    "Remember to save the file before closing.",
]
save_vectors = model.encode(save_sentences)
scores = cosine_similarity(save_vectors)[0]
for text, score in zip(save_sentences[1:], scores[1:]):
    print(f"{score:.2f}  {text}")

✏️ **Try it:** See if you can do each of these, and keep track of your best example of the model getting it wrong. We'll share some at the end.

1. Find two sentences with **no words in common** that score high.
2. Find two sentences with **nearly identical words** that mean opposite things. Does the model notice?
3. Try something from **your own field**, or another word with several meanings.
4. Try **another language**. This small model was trained mostly on English. What happens to a sentence and its translation?

In [ ]:
compare(
    "The whale swam beneath the ship.", # ✏️ change this text
    "A huge sea creature passed under the boat.", # ✏️ change this text
)

## Your Turn
These activities are for after the session, or if you finish early. Each one uses the same few functions that we've built.

### Activity 1: Choose Your Collection
Which books you include changes every score. Choose your own collection from our eight books. Then pick one book and see how its top words differ from what they were with all eight.

- Which words moved up or down? Why?
- What collection would you build to bring out something about this book other than its characters?

In [ ]:
my_collection = ["Dracula", "Frankenstein", "Sherlock Holmes"] # ✏️ choose at least two books
my_book = "Dracula" # ✏️ choose one of them

print("In your collection:", list(top_words(tf_idf(book_counts.loc[my_collection]), my_book).index))
print("In all eight books:", list(top_words(book_scores, my_book).index))

### Activity 2: Your Own Documents
Paste in a few short documents of your own: abstracts, poems, song lyrics, or emails. Use at least three. Before you run the cell, guess which two are the most alike and what each document's top word will be.

- Which words do the most to make your closest pair look alike? Do they reflect what you'd say the documents have in common?

In [ ]:
my_documents = {
    "first": "Paste your first document here.", # ✏️ replace these with your own
    "second": "Paste your second document here.",
    "third": "And paste a third document here, along with others if you like.",
}
my_scores = tf_idf(bag_of_words(my_documents))

for name in my_documents:
    print(name, list(top_words(my_scores, name, 5).index))
similarity_table(my_scores).round(2)

### Activity 3: Read the Neighbors
Pick a genre, and look at the closest pairs of documents within the whole Brown Corpus. Then read the opening of each pair.

- Why were they matched? Check with `shared_words`. Would a person call them similar?

In [ ]:
my_genre = "religion" # ✏️ try "news", "mystery", "hobbies", "romance", or another genre

brown_scores = tf_idf(brown_counts)
genre_documents = [doc for doc in brown_counts.index if brown_genre[doc] == my_genre]
closest = similarity_table(brown_scores.loc[genre_documents]).where(lambda table: table < 0.999).stack().nlargest(1)
first, second = closest.index[0]

print(f"Closest pair in {my_genre}: {first} and {second} (similarity {closest.iloc[0]:.2f})")
print("Words that link them:", list(shared_words(brown_scores, first, second).index))
print("\n" + first + ":", " ".join(brown.words(first)[:40]), "...")
print("\n" + second + ":", " ".join(brown.words(second)[:40]), "...")

## Wrapping Up: What Does the Math See?
Back to where we began: how would you compare a thousand documents? Has your answer changed?

These methods don't read for meaning. They measure patterns that sometimes line up with it, and every choice shapes what they can see. Before you go, think back over the choices we made:

- **The bag:** What did we lose when we threw out word order? When would that matter to you, and when wouldn't it?
- **TF-IDF:** What did "distinctive" end up meaning? What's distinctive about a book that isn't a name?
- **Similarity:** When did a high score tell you something, and when did it just reflect a character named Elizabeth?
- **The collection:** Who chose what was in it, and what would you add or leave out?
- **Embeddings:** What was your best example of the model getting it wrong? Who decided what the model learned from?

**One more thing to think about:** Next time, in *Making Text*, we'll look at language generation. The models behind it are built on the same idea as our embeddings: representing language as numbers learned from patterns of use.

### Places to Go Next
- [Voyant Tools](https://voyant-tools.org/): Explore word counts and patterns in your own texts, with no code
- [Google Books Ngram Viewer](https://books.google.com/ngrams): Chart words and phrases across millions of books

**Bonus:** Try a word in your field on the Ngram Viewer. Which senses of it might one curve be hiding?